# KweriMoto: YAMNet baseline

**Frozen YAMNet features + a small three-class classifier** (healthy / engine_knock / misfire).

[Open in Colab](https://colab.research.google.com/github/SUUNARAYMOND/kwerimoto-dataset/blob/main/notebooks/yamnet_baseline.ipynb) · Dataset: [rayyyyy007/kwerimoto-engine-sounds](https://huggingface.co/datasets/rayyyyy007/kwerimoto-engine-sounds)

```
clip (16 kHz) → YAMNet (frozen) → 1,024 numbers per 0.96 s piece → combine per clip → small classifier → label
```

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Load the dataset |
| 2 | Look at the data |
| 3 | Get YAMNet features |
| 4 | Sanity check: what does YAMNet hear? |
| 5 | Train the head |
| 6 | Test (once) |
| 7 | `real_test` |
| 8 | Check the mistakes |
| 9 | Save |

**Rules:** train on `train`, choose settings on `validation`, score `test` once at the end, never train or tune on `real_test`.

## 0. Setup

Fixed settings in one place. `SEED` makes random choices repeat on every run. On Colab, results are saved
to your Google Drive so they survive when the session ends.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

DATASET_ID = "rayyyyy007/kwerimoto-engine-sounds"
LABELS = ["healthy", "engine_knock", "misfire"]
SAMPLE_RATE = 16_000
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:  # on Colab: save to Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/kwerimoto/yamnet")
except ImportError:  # anywhere else: save next to the notebook
    SAVE_DIR = Path("outputs/yamnet")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Saving results to", SAVE_DIR)

## 1. Load the dataset

`snapshot_download` copies the dataset from Hugging Face (about 290 MB, once per session) and returns the folder.
`manifest.csv` has one row per clip: its file, label, split and source. We add `path`, the full path to each clip.

In [ ]:
from huggingface_hub import snapshot_download

root = Path(snapshot_download(DATASET_ID, repo_type="dataset"))
manifest = pd.read_csv(root / "manifest.csv", keep_default_na=False)
manifest["path"] = [str(root / f) for f in manifest["file_name"]]
print(len(manifest), "clips")
manifest.head(3)

## 2. Look at the data

Before training anything, check what we have: how many clips per label and split, and how long they are.

In [ ]:
pd.crosstab([manifest["label"], manifest["source_dataset"]], manifest["split"], margins=True, margins_name="total")[
    ["train", "validation", "test", "real_test", "total"]]

**Clip lengths.** YAMNet looks at 0.96 s pieces, so a clip shorter than 0.96 s gives a single piece that is
partly silence. Most of these short clips are synthetic knock, which is why we check later whether the model's
mistakes depend on length.

In [ ]:
display(manifest.groupby(["label", "source_dataset"])["duration_s"].describe()[["count", "min", "50%", "max"]].round(2))

short = manifest[manifest["duration_s"] < 0.96]
print(f"{len(short)} clips are shorter than one YAMNet piece (0.96 s):", short.groupby("label").size().to_dict())

fig, ax = plt.subplots(figsize=(8, 3))
for label in LABELS:
    ax.hist(manifest.loc[manifest["label"] == label, "duration_s"], bins=40, alpha=0.6, label=label)
ax.set_xlabel("clip length (s)"); ax.set_ylabel("clips"); ax.legend(); plt.show()

**Listen.** One random training clip per label.

In [ ]:
for label in LABELS:
    row = manifest[(manifest["split"] == "train") & (manifest["label"] == label)].sample(1, random_state=SEED).iloc[0]
    print(f"{label}: {row['file_name']} ({row['duration_s']} s, from {row['source_dataset']})")
    display(Audio(row["path"]))

## 3. Get YAMNet features

Load Google's YAMNet. It is **frozen**: we only use it, never train it.

For each clip YAMNet returns, per 0.96 s piece (a new piece every 0.48 s):
- **embedding**: 1,024 numbers describing how the piece sounds; this is what our classifier learns from;
- **scores**: how likely each of YAMNet's 521 sound types is (we use these in section 4).

Our labels are per clip, so we combine the pieces of each clip in two ways and later keep whichever works
better on `validation`:
- **mean**: the average piece, which describes the clip as a whole;
- **max**: the strongest value of each number over the pieces, which keeps short events (like a knock) that
  averaging could wash out.

This runs once (about 1 to 3 minutes on CPU) and is saved to Drive; later runs load the saved file.

In [ ]:
import subprocess
import sys

import tensorflow as tf
from tqdm.auto import tqdm

try:
    import kagglehub
except ImportError:  # Colab normally has it; install if missing
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

# Google's official YAMNet (formerly on TF Hub, now hosted on Kaggle)
yamnet = tf.saved_model.load(kagglehub.model_download("google/yamnet/tensorFlow2/yamnet"))
class_names = pd.read_csv(yamnet.class_map_path().numpy().decode())["display_name"].tolist()
print(len(class_names), "YAMNet sound types, e.g.", class_names[:3])


def yamnet_features(path):
    audio, sr = sf.read(path, dtype="float32")
    assert sr == SAMPLE_RATE and audio.ndim == 1, "YAMNet needs mono 16 kHz audio"
    scores, embeddings, _ = yamnet(audio)
    scores, embeddings = scores.numpy(), embeddings.numpy()
    return {"emb_mean": embeddings.mean(axis=0), "emb_max": embeddings.max(axis=0),
            "score_mean": scores.mean(axis=0), "score_max": scores.max(axis=0), "n_pieces": len(embeddings)}

In [ ]:
FEATURES = SAVE_DIR / "yamnet_features.npz"
file_names = manifest["file_name"].to_numpy(dtype=str)

if FEATURES.exists() and np.array_equal(np.load(FEATURES)["file_name"], file_names):
    feats = dict(np.load(FEATURES))
    print("Loaded saved features from", FEATURES)
else:
    rows = [yamnet_features(path) for path in tqdm(manifest["path"], desc="YAMNet")]
    feats = {key: np.stack([row[key] for row in rows]) for key in rows[0]}
    feats["file_name"] = file_names
    np.savez_compressed(FEATURES, **feats)
    print("Saved features to", FEATURES)

for key in ["emb_mean", "emb_max", "score_mean", "n_pieces"]:
    print(f"{key:10s} {feats[key].shape}")
manifest["n_pieces"] = feats["n_pieces"]
manifest.groupby("label")["n_pieces"].describe()[["min", "50%", "max"]]

## 4. Sanity check: what does YAMNet hear?

Before training, see what YAMNet itself recognises in our clips. Only `train` clips are used, so the other
splits stay untouched.

**Top sound types per label**: YAMNet's five highest average scores for each label. We hope to see engine and
vehicle sounds.

In [ ]:
train = (manifest["split"] == "train").to_numpy()
for label in LABELS:
    average = feats["score_mean"][train & (manifest["label"] == label).to_numpy()].mean(axis=0)
    top = np.argsort(average)[::-1][:5]
    print(f"{label:13s}", ", ".join(f"{class_names[i]} ({average[i]:.2f})" for i in top))

**YAMNet's own "Engine knocking" score.** YAMNet already knows a sound type called "Engine knocking". For each
clip we take its highest knock score over the pieces (0 = not heard, 1 = sure). If knock clips score clearly
higher than the others, YAMNet already hears knocking without any training from us. In section 6 we test this
properly as a no-training comparison for our classifier.

In [ ]:
KNOCK = class_names.index("Engine knocking")
manifest["yamnet_knock"] = feats["score_max"][:, KNOCK]
manifest[train].groupby(["label", "source_dataset"])["yamnet_knock"].describe()[["count", "25%", "50%", "75%"]].round(3)